# Save Your Work

Before starting, save this notebook to your Google Drive:
1. Click **File** → **Save a copy in Drive**
2. The copy will open automatically
3. Work in the Google Drive copy from now on

---

# Census Income Case Study

Apply the full data preparation workflow — EDA, cleaning, and feature engineering — to the 1994 US Census Income Dataset, then preview the classifier you'll train in module 09.

This notebook covers all four lessons in module 08:
1. Introduction & Data Loading
2. Exploratory Data Analysis
3. Data Cleaning
4. Feature Engineering

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

---

# Part 1: Introduction & Data Loading

The **Census Income Dataset** (UCI Adult) was extracted from the 1994 US Census Bureau database. Each row describes one person from the survey. The business question:

> **Can we predict whether a person earns more than $50,000 per year from demographic and employment information alone?**

The raw file has no header row — column names must be provided manually.

## Load the Dataset

In [ ]:
columns = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week", "native_country", "income"
]

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
df = pd.read_csv(url, header=None, names=columns, sep=", ", engine="python")

print(f"Shape: {df.shape}")
print(f"\nData types:")
print(df.dtypes)

## Initial Impressions

In [ ]:
# Class distribution
print("Income class distribution:")
print(df["income"].value_counts())
print(f"\nHigh-income fraction: {(df['income'] == '>50K').mean():.1%}")

In [ ]:
# Standard NaN check
print(f"Standard NaN values: {df.isnull().sum().sum()}")

# Check for '?' placeholder
print("\nCount of '?' per column:")
print((df == "?").sum()[(df == "?").sum() > 0])

In [ ]:
df.describe()

**Initial scan summary:**
- 32,561 rows × 15 columns; 6 numeric, 9 string
- 76% earn ≤$50K, 24% earn >$50K — moderately imbalanced
- `?` in `workclass` (1,836), `occupation` (1,843), `native_country` (583)
- `fnlwgt` is a census sampling weight — not a predictive feature
- `education` and `education_num` encode the same information

---

# Part 2: Exploratory Data Analysis

Before changing anything, understand which features matter and why.

In [ ]:
# Add a numeric label for easier analysis
df["label"] = (df["income"] == ">50K").astype(int)   # 1 = high income, 0 = low income

## Step 1: Age Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df["age"], bins=40, color="steelblue", edgecolor="white")
axes[0].set_title("Age Distribution (All)")
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Count")

axes[1].hist(df[df["label"] == 0]["age"], bins=40, alpha=0.6,
             color="#2196f3", label="<=50K")
axes[1].hist(df[df["label"] == 1]["age"], bins=40, alpha=0.6,
             color="#e74c3c", label=">50K")
axes[1].set_title("Age Distribution by Income")
axes[1].set_xlabel("Age")
axes[1].legend()

plt.tight_layout()
plt.show()

print(df.groupby("income")["age"].agg(["mean", "median"]).round(1))

## Step 2: Income Rate by Education

In [ ]:
# Sort education levels by education_num (years of schooling)
edu_order = (
    df.groupby("education")["education_num"]
    .mean()
    .sort_values()
    .index
    .tolist()
)

edu_rates = (
    df.groupby("education")["label"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "income_rate", "count": "n"})
    .loc[edu_order]
    .round(3)
)

print(edu_rates.to_string())

In [ ]:
plt.figure(figsize=(11, 5))
income_rates = edu_rates["income_rate"]
colors = ["#e74c3c" if r > 0.5 else "#2196f3" for r in income_rates]
plt.bar(range(len(edu_order)), income_rates, color=colors, tick_label=edu_order)
plt.xticks(rotation=45, ha="right")
plt.title("Income Rate (>$50K) by Education Level")
plt.ylabel("Fraction earning >$50K")
plt.axhline(0.5, color="black", linestyle="--", linewidth=0.8)
plt.tight_layout()
plt.show()

## Step 3: Income Rate by Occupation

In [ ]:
occ_rates = (
    df[df["occupation"] != "?"]
    .groupby("occupation")["label"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "income_rate", "count": "n"})
    .sort_values("income_rate", ascending=False)
    .round(3)
)

print(occ_rates.to_string())

## Step 4: Hours Worked Per Week

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df[df["label"] == 0]["hours_per_week"], bins=40, alpha=0.6,
             color="#2196f3", label="<=50K")
axes[0].hist(df[df["label"] == 1]["hours_per_week"], bins=40, alpha=0.6,
             color="#e74c3c", label=">50K")
axes[0].set_title("Hours/Week by Income")
axes[0].set_xlabel("Hours per week")
axes[0].legend()

axes[1].boxplot(
    [df[df["label"] == 0]["hours_per_week"],
     df[df["label"] == 1]["hours_per_week"]],
    labels=["<=50K", ">50K"],
    patch_artist=True,
    boxprops=dict(facecolor="lightblue"),
)
axes[1].set_title("Hours/Week by Income (Box Plot)")
axes[1].set_ylabel("Hours per week")

plt.tight_layout()
plt.show()

print(df.groupby("income")["hours_per_week"].agg(["mean", "median"]).round(1))

## Step 5: Capital Gain and Capital Loss

In [ ]:
print("Capital gain distribution:")
print(f"  Fraction with zero: {(df['capital_gain'] == 0).mean():.1%}")
print(f"  Among non-zero — median: {df[df['capital_gain'] > 0]['capital_gain'].median():.0f}")
print(f"  Among non-zero — max: {df['capital_gain'].max()}")
print()
print("Capital loss distribution:")
print(f"  Fraction with zero: {(df['capital_loss'] == 0).mean():.1%}")

# Income rate for those with vs without capital activity
has_cap = (df["capital_gain"] > 0) | (df["capital_loss"] > 0)
print(f"\nIncome rate — with capital activity:     {df[has_cap]['label'].mean():.1%}")
print(f"Income rate — without capital activity:  {df[~has_cap]['label'].mean():.1%}")

## Step 6: Income Rate by Workclass

In [ ]:
wc_rates = (
    df[df["workclass"] != "?"]
    .groupby("workclass")["label"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "income_rate", "count": "n"})
    .sort_values("income_rate", ascending=False)
    .round(3)
)

print(wc_rates.to_string())

## Step 7: Correlation Matrix

In [ ]:
numeric_cols = ["age", "fnlwgt", "education_num", "capital_gain",
                "capital_loss", "hours_per_week", "label"]

plt.figure(figsize=(7, 6))
sns.heatmap(
    df[numeric_cols].corr(),
    annot=True, fmt=".2f", cmap="coolwarm",
    vmin=-1, vmax=1, linewidths=0.5,
)
plt.title("Correlation Matrix — Numeric Features")
plt.tight_layout()
plt.show()

print("\nCorrelation with income (label):")
print(df[numeric_cols].corr()["label"].drop("label").sort_values(ascending=False).round(3))

## Step 8: Native Country

In [ ]:
print(f"Unique native countries: {df['native_country'].nunique()}")
print(f"\nTop 5 most common:")
print(df["native_country"].value_counts().head())
print(f"\nUnited-States fraction: {(df['native_country'] == 'United-States').mean():.1%}")

**EDA Summary:**

| Finding | Implication |
|---------|-------------|
| 76%/24% class split | Accuracy is misleading; evaluate with precision, recall, and AUC |
| Age: median gap of 10 years between income groups | Strong feature |
| Education: near-monotonic 0%→73% income rate gradient | `education_num` captures this perfectly |
| Capital gain/loss: 91%+ zero but 63% high-income rate when non-zero | Create `has_capital` flag |
| `fnlwgt`: r = −0.008 with income | Drop unconditionally |
| `native_country`: 89.6% US, 41 values | Create `native_us` binary flag |

---

# Part 3: Data Cleaning

Fix every issue identified in EDA, with a documented reason for each decision.

In [ ]:
# Always work on a copy to preserve the raw data
df_clean = df.drop(columns=["label"]).copy()
print(f"Starting shape: {df_clean.shape}")

## Step 1: Remove Duplicate Rows

In [ ]:
n_dupes = df_clean.duplicated().sum()
print(f"Duplicate rows: {n_dupes}")

df_clean = df_clean.drop_duplicates()
print(f"Shape after deduplication: {df_clean.shape}")

## Step 2: Drop fnlwgt

In [ ]:
df_clean = df_clean.drop(columns=["fnlwgt"])
print(f"Shape after dropping fnlwgt: {df_clean.shape}")

## Step 3: Resolve the education / education_num Redundancy

In [ ]:
# Confirm that education and education_num are perfectly redundant
edu_mapping = df_clean.groupby("education")["education_num"].agg(["min", "max"])
print("All min==max means perfect 1:1 mapping:")
print((edu_mapping["min"] == edu_mapping["max"]).all())

# Drop the string column; keep the numeric ordinal
df_clean = df_clean.drop(columns=["education"])
print(f"Shape after dropping education: {df_clean.shape}")

## Step 4: Handle ? in workclass

In [ ]:
print(f"workclass '?' count: {(df_clean['workclass'] == '?').sum()}")

# Treat '?' as a distinct category — these individuals are not in the labor force
df_clean["workclass"] = df_clean["workclass"].replace("?", "Unknown")
print(f"workclass values: {sorted(df_clean['workclass'].unique())}")

## Step 5: Handle ? in occupation

In [ ]:
print(f"occupation '?' count: {(df_clean['occupation'] == '?').sum()}")

df_clean["occupation"] = df_clean["occupation"].replace("?", "Unknown")
print(f"occupation '?' remaining: {(df_clean['occupation'] == '?').sum()}")

## Step 6: Final Validation

In [ ]:
print("=== Cleaned Dataset ===")
print(f"Shape:                    {df_clean.shape}")
print(f"Columns:                  {list(df_clean.columns)}")
print(f"Standard NaN values:      {df_clean.isnull().sum().sum()}")
print(f"Remaining '?' anywhere:   {(df_clean == '?').sum().sum()}  (in native_country only — handled in feature engineering)")

**Cleaning summary:**

| Step | Change | Rationale |
|------|--------|----------|
| Remove duplicates | −24 rows | 24 identical rows, almost certainly data entry errors |
| Drop `fnlwgt` | −1 column | Census sampling weight; r = −0.008 with income |
| Drop `education` | −1 column | Perfectly redundant with `education_num` |
| Replace `?` with `Unknown` in `workclass` | 1,836 values | Non-labor-force status is a meaningful category |
| Replace `?` with `Unknown` in `occupation` | 1,843 values | Same rationale |
| Leave `native_country` `?` untouched | — | Will become `0` in the `native_us` binary feature |

---

# Part 4: Feature Engineering

Transform the cleaned data into a fully numeric feature matrix ready for a classifier.

In [ ]:
df_fe = df_clean.copy()
print(f"Starting shape: {df_fe.shape}")

## Step 1: Encode the Target

In [ ]:
df_fe["income"] = (df_fe["income"] == ">50K").astype(int)

print("Target distribution after encoding:")
print(df_fe["income"].value_counts())
print(f"High-income rate: {df_fe['income'].mean():.1%}")

## Step 2: Train/Test Split First

**Always split before feature engineering.** Fit all transformers on training data only.

In [ ]:
X = df_fe.drop(columns=["income"])
y = df_fe["income"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"High-income rate — train: {y_train.mean():.1%}, test: {y_test.mean():.1%}")

## Step 3: Encode sex as a Binary Feature

In [ ]:
X_train = X_train.copy()
X_test  = X_test.copy()

X_train["sex"] = (X_train["sex"] == "Male").astype(int)
X_test["sex"]  = (X_test["sex"] == "Male").astype(int)

print(f"sex value counts (train): {X_train['sex'].value_counts().to_dict()}")

## Step 4: Handle native_country — Create native_us

In [ ]:
X_train["native_us"] = (X_train["native_country"] == "United-States").astype(int)
X_test["native_us"]  = (X_test["native_country"] == "United-States").astype(int)

print(f"native_us distribution (train):")
print(X_train["native_us"].value_counts())

X_train = X_train.drop(columns=["native_country"])
X_test  = X_test.drop(columns=["native_country"])

## Step 5: Handle capital_gain and capital_loss — Create has_capital

In [ ]:
X_train["has_capital"] = (
    (X_train["capital_gain"] > 0) | (X_train["capital_loss"] > 0)
).astype(int)

X_test["has_capital"] = (
    (X_test["capital_gain"] > 0) | (X_test["capital_loss"] > 0)
).astype(int)

print(f"has_capital distribution (train): {X_train['has_capital'].value_counts().to_dict()}")
print(f"Fraction with capital activity: {X_train['has_capital'].mean():.1%}")

## Step 6: One-Hot Encode Categorical Features

In [ ]:
cat_cols = ["workclass", "marital_status", "occupation", "relationship", "race"]

# Fit on training data only
X_train_enc = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
X_test_enc  = pd.get_dummies(X_test,  columns=cat_cols, drop_first=True)

# Align columns (test set may be missing categories unseen in training)
X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

print(f"Shape after one-hot encoding:")
print(f"  X_train_enc: {X_train_enc.shape}")
print(f"  X_test_enc:  {X_test_enc.shape}")

## Step 7: Scale Numeric Features

In [ ]:
scale_cols = ["age", "education_num", "hours_per_week", "capital_gain", "capital_loss"]

scaler = StandardScaler()
X_train_enc[scale_cols] = scaler.fit_transform(X_train_enc[scale_cols])
X_test_enc[scale_cols]  = scaler.transform(X_test_enc[scale_cols])

print("Scaled column statistics (train, should be mean≈0, std≈1):")
print(X_train_enc[scale_cols].describe().loc[["mean", "std"]].round(3))

## Step 8: Sanity Check

In [ ]:
assert X_train_enc.isnull().sum().sum() == 0, "Unexpected NaN in training features"
assert X_test_enc.isnull().sum().sum() == 0, "Unexpected NaN in test features"
assert X_train_enc.shape[1] == X_test_enc.shape[1], "Train/test column mismatch"

print("All checks passed.")
print(f"\nFinal feature matrix shapes:")
print(f"  X_train_enc: {X_train_enc.shape}")
print(f"  X_test_enc:  {X_test_enc.shape}")
print(f"\nTotal features: {X_train_enc.shape[1]}")

**Feature Engineering Summary:**

| Step | Decision | Rationale |
|------|----------|----------|
| Encode target | `<=50K` → 0, `>50K` → 1 | Positive class = high income |
| Train/test split first | 80/20, `stratify=y` | Prevents data leakage; preserves class balance |
| Binary encode `sex` | Male=1, Female=0 | Two categories; no need for one-hot |
| Create `native_us` | 1 if United-States, 0 otherwise | Collapses 41 high-cardinality values cleanly |
| Create `has_capital` | 1 if any capital activity, 0 otherwise | Captures 63% vs 18% high-income signal |
| One-hot encode 5 categoricals | `drop_first=True` | Avoids multicollinearity |
| Scale 5 continuous columns | `StandardScaler`, fit on train only | Consistent scale for distance/gradient algorithms |

---

# Preview: Binary Classification (Module 09)

The feature matrix is ready. Here is a preview of what module 09 will build on it.

In [ ]:
lr = LogisticRegression(max_iter=500, random_state=42)
lr.fit(X_train_enc, y_train)

y_pred  = lr.predict(X_test_enc)
y_proba = lr.predict_proba(X_test_enc)[:, 1]

print(classification_report(y_test, y_pred, target_names=["<=50K", ">50K"]))
print(f"AUC: {roc_auc_score(y_test, y_proba):.4f}")

Logistic regression achieves ~85% accuracy and AUC ~0.91 with no hyperparameter tuning. Notice that recall on the `>50K` class is lower than on `<=50K` — this is the class imbalance effect. Module 09 will explore techniques for improving recall on the minority class.